## **Laboratorio: Dashboard de Datos Interactivo con Streamlit**

Streamlit es una librería de Python de código abierto que permite crear aplicaciones web interactivas para ciencia de datos y Machine Learning con muy pocas líneas de código. A diferencia de frameworks como Flask o Django, Streamlit no requiere conocimientos de HTML, CSS ni JavaScript: todo se construye en Python puro.

Un **dashboard de datos** es una interfaz visual que permite explorar, filtrar y analizar un conjunto de datos de forma interactiva. Los dashboards son herramientas fundamentales en el flujo de trabajo de un Data Scientist para:
- Comunicar hallazgos a stakeholders no técnicos
- Explorar datasets durante el EDA (*Exploratory Data Analysis*)
- Monitorizar el rendimiento de modelos en producción
- Crear prototipos rápidos de aplicaciones de ML

En este laboratorio construirás paso a paso un dashboard completo para explorar el dataset **Palmer Penguins**, que contiene medidas morfológicas de tres especies de pingüinos del Archipiélago Palmer, Antártida.

## ¿Cómo funciona este laboratorio?

A diferencia de los laboratorios anteriores, Streamlit **no puede ejecutarse dentro del kernel de Jupyter**. Streamlit es un servidor web independiente que se lanza desde la terminal con el comando `streamlit run dashboard.py`.

Para que puedas seguir el laboratorio de forma progresiva, usaremos la magia `%%writefile` de Jupyter:

- La **Celda 3** usa `%%writefile dashboard.py` para **crear** el archivo desde cero.
- Las celdas siguientes usan `%%writefile -a dashboard.py` para **añadir** código al archivo.
- En varios puntos encontrarás un **🔁 Checkpoint** indicando que debes ejecutar la app en tu terminal para ver el resultado.

**Instrucción importante:** Abre una terminal en la misma carpeta que este notebook y déjala abierta durante todo el laboratorio.

> ⚠️ **Atención:** Si vuelves a ejecutar la Celda 3 (que usa `%%writefile` sin `-a`), el archivo `dashboard.py` se sobreescribirá desde el principio. Si quieres empezar de cero, ejecuta todas las celdas en orden desde la Celda 3.

## Objetivos del Laboratorio

Al finalizar este laboratorio habrás aprendido a:

1. **Instalar y configurar** Streamlit en un entorno local.
2. **Usar componentes de texto y estructura** para construir el esqueleto de una app.
3. **Construir widgets interactivos** en la barra lateral para filtrar datos en tiempo real.
4. **Crear visualizaciones dinámicas** con Plotly que responden a los filtros del usuario.
5. **Publicar métricas resumidas** con el componente `st.metric`.
6. **Entender el modelo de ejecución** de Streamlit y cómo gestionar el estado con `st.session_state`.

**Duración estimada:** 2-3 horas

---

## Sección 1: Preparación del Entorno

**Celda 1: Instalación de Dependencias**

In [ ]:
# Instalar las librerías necesarias para el laboratorio
# Si ya las tienes instaladas, comenta las líneas correspondientes
!pip install streamlit seaborn pandas plotly openai

**Explicación:**

- `streamlit` — el framework principal para construir la app web
- `seaborn` — librería de visualización que incluye datasets de ejemplo (usaremos Palmer Penguins)
- `pandas` — manipulación y análisis de datos tabulares
- `plotly` — librería de gráficos interactivos con soporte nativo en Streamlit

**Celda 2: Verificación de la Instalación**

In [ ]:
import streamlit
import seaborn as sns
import pandas as pd
import plotly.express as px

print(f"Streamlit versión: {streamlit.__version__}")
print(f"Pandas versión: {pd.__version__}")

# Cargar el dataset Palmer Penguins
df = sns.load_dataset('penguins')
print(f"\nDataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()

**Explicación del Dataset: Palmer Penguins**

El dataset contiene **344 observaciones** de pingüinos de tres islas del Archipiélago Palmer, Antártida. Las variables son:

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `species` | Categórica | Especie: *Adelie*, *Chinstrap* o *Gentoo* |
| `island` | Categórica | Isla: *Biscoe*, *Dream* o *Torgersen* |
| `bill_length_mm` | Numérica | Longitud del pico (mm) |
| `bill_depth_mm` | Numérica | Profundidad del pico (mm) |
| `flipper_length_mm` | Numérica | Longitud de la aleta (mm) |
| `body_mass_g` | Numérica | Masa corporal (g) |
| `sex` | Categórica | Sexo: *male* o *female* |

Es un dataset ideal para dashboards: tiene variables categóricas que sirven como filtros y variables numéricas que permiten comparaciones visuales entre grupos.

**Celda 3: Creación del Archivo Base del Dashboard**

In [ ]:
%%writefile dashboard.py
# ============================================================
# Dashboard de Análisis de Pingüinos - Palmer Archipelago
# Laboratorio Streamlit - Máster en IA / Deep Learning
# ============================================================

import streamlit as st
import seaborn as sns
import pandas as pd
import plotly.express as px

# --- Configuración de la página ---
# set_page_config DEBE ser la primera llamada a Streamlit en el script
st.set_page_config(
    page_title="Dashboard de Pingüinos",
    page_icon="🐧",
    layout="wide"   # 'wide' usa todo el ancho de la pantalla
)

# --- Carga de datos con caché ---
@st.cache_data
def cargar_datos():
    """Carga el dataset de pingüinos y elimina filas con valores nulos."""
    df = sns.load_dataset('penguins')
    df = df.dropna()   # eliminamos las 11 filas con valores nulos
    return df

df = cargar_datos()

**Explicación:**

**`st.set_page_config`** — Configura el título de la pestaña del navegador, el icono y el layout. `layout="wide"` es esencial para dashboards ya que aprovecha todo el ancho de la pantalla. Esta función debe ser **la primera llamada a Streamlit** en el script, de lo contrario lanzará un error.

**`@st.cache_data`** — Este decorador es uno de los más importantes en Streamlit. Sin él, el script completo se re-ejecuta de arriba a abajo cada vez que el usuario interactúa con un widget (cambia un filtro, mueve un slider...). Esto significa que la función `cargar_datos()` se llamaría cientos de veces. Con `@st.cache_data`, Streamlit guarda el resultado en memoria y solo lo recalcula si los argumentos de la función cambian. En datasets grandes, esto puede suponer la diferencia entre una app fluida y una inutilizable.

---
## Sección 2: Componentes de Texto y Encabezado del Dashboard

**Celda 4: Título, Subtítulo y Texto Descriptivo**

In [ ]:
%%writefile -a dashboard.py

# --- Encabezado principal del dashboard ---
st.title("🐧 Dashboard de Análisis de Pingüinos")
st.subheader("Exploración del Dataset Palmer Penguins")
st.markdown("""
Este dashboard permite explorar las características morfológicas de tres especies
de pingüinos (*Adélie*, *Chinstrap* y *Gentoo*) de las Islas Palmer, Antártida.
Utiliza los filtros del panel lateral izquierdo para segmentar los datos y observa
cómo se actualizan los gráficos y métricas en tiempo real.
""")
st.divider()   # línea horizontal divisora

**Explicación — Componentes de texto en Streamlit:**

| Función | Uso |
|---------|-----|
| `st.title()` | Título principal de la página (H1) |
| `st.header()` | Encabezado de sección (H2) |
| `st.subheader()` | Sub-encabezado (H3) |
| `st.markdown()` | Texto con formato Markdown completo (negrita, cursiva, listas, links) |
| `st.text()` | Texto plano sin formato |
| `st.caption()` | Texto secundario pequeño, ideal para notas y atribuciones |
| `st.divider()` | Línea horizontal para separar secciones visualmente |

`st.markdown()` acepta el mismo Markdown que usas en los notebooks, incluyendo HTML embebido si necesitas mayor control sobre el estilo.

### 🔁 Checkpoint 1 — Primera ejecución

Abre una terminal en la carpeta donde está `dashboard.py` y ejecuta:

```bash
streamlit run dashboard.py
```

Streamlit abrirá automáticamente tu navegador en `http://localhost:8501`. Deberías ver el título, subtítulo y texto descriptivo del dashboard.

**Deja la terminal abierta** durante el resto del laboratorio. Cada vez que ejecutes una celda `%%writefile -a`, el archivo se actualiza y Streamlit recargará la app automáticamente.

---

## Sección 3: Barra Lateral y Widgets Interactivos

La barra lateral (`st.sidebar`) es el lugar natural para colocar los controles de filtrado en un dashboard. Todos los componentes de Streamlit tienen una versión sidebar accesible con el prefijo `st.sidebar.` (por ejemplo, `st.sidebar.multiselect()`). Cuando el usuario interactúa con cualquier widget, Streamlit re-ejecuta el script completo de arriba a abajo con los nuevos valores, actualizando automáticamente todos los gráficos y métricas.

**Celda 5: Filtros Categóricos en la Barra Lateral**

In [ ]:
%%writefile -a dashboard.py

# --- Barra lateral: filtros ---
st.sidebar.header("⚙️ Filtros")
st.sidebar.markdown("Selecciona los subconjuntos de datos que deseas visualizar.")

# Filtro por especie (multiselect: el usuario puede elegir una o varias)
especies_disponibles = sorted(df['species'].unique().tolist())
especies_seleccionadas = st.sidebar.multiselect(
    label="Especie",
    options=especies_disponibles,
    default=especies_disponibles   # por defecto todas seleccionadas
)

# Filtro por isla
islas_disponibles = sorted(df['island'].unique().tolist())
islas_seleccionadas = st.sidebar.multiselect(
    label="Isla",
    options=islas_disponibles,
    default=islas_disponibles
)

# Filtro por sexo
sexos_disponibles = sorted(df['sex'].unique().tolist())
sexo_seleccionado = st.sidebar.multiselect(
    label="Sexo",
    options=sexos_disponibles,
    default=sexos_disponibles
)

**Explicación — `st.multiselect`:**

`st.multiselect(label, options, default)` muestra un desplegable donde el usuario puede seleccionar **múltiples opciones**. Siempre devuelve una **lista de Python** con los valores seleccionados. Los parámetros principales son:
- `label` — texto visible encima del widget
- `options` — lista de valores posibles (strings, enteros, etc.)
- `default` — valor(es) seleccionados por defecto al cargar la app

Usamos `sorted()` para que las opciones aparezcan en orden alfabético, haciendo la UI más predecible.

**Celda 6: Filtro Numérico con Slider de Rango**

In [ ]:
%%writefile -a dashboard.py

# Filtro numérico: rango de masa corporal
masa_min = int(df['body_mass_g'].min())
masa_max = int(df['body_mass_g'].max())

rango_masa = st.sidebar.slider(
    label="Masa Corporal (g)",
    min_value=masa_min,
    max_value=masa_max,
    value=(masa_min, masa_max)   # tupla de dos valores → slider de rango
)

st.sidebar.divider()
# Información sobre el tamaño original del dataset
st.sidebar.info(f"Dataset original: **{len(df)}** pingüinos (sin valores nulos)")

**Explicación — `st.slider` de rango:**

Cuando `value` es una **tupla de dos elementos** `(min, max)`, `st.slider` crea un **slider de rango** con dos manejadores. El widget devuelve una tupla `(valor_inferior, valor_superior)` que usaremos para filtrar el DataFrame.

Convertimos los valores a `int` con `int()` porque los valores mínimo y máximo del DataFrame son `float64` y el slider los mostraría con decimales innecesarios.

**`st.sidebar.info()`** muestra un recuadro informativo azul — existen variantes para otros estilos: `st.success()` (verde), `st.warning()` (amarillo) y `st.error()` (rojo).

**Celda 7: Aplicación de los Filtros al DataFrame**

In [ ]:
%%writefile -a dashboard.py

# --- Aplicar los filtros seleccionados al DataFrame ---
df_filtrado = df[
    (df['species'].isin(especies_seleccionadas)) &
    (df['island'].isin(islas_seleccionadas)) &
    (df['sex'].isin(sexo_seleccionado)) &
    (df['body_mass_g'] >= rango_masa[0]) &
    (df['body_mass_g'] <= rango_masa[1])
]

# Mostrar advertencia y detener la app si no hay datos con los filtros aplicados
if df_filtrado.empty:
    st.warning("⚠️ No hay datos con los filtros seleccionados. Ajusta los filtros del panel lateral.")
    st.stop()   # detiene la ejecución del resto del script de forma limpia

**Explicación — Patrón de filtrado en pandas:**

Combinamos múltiples condiciones booleanas con el operador `&` (AND). El método `isin()` verifica si cada valor de la columna está dentro de la lista retornada por el `st.multiselect`.

**`st.stop()`** es una función muy útil: detiene la ejecución del script en ese punto sin lanzar un error. Úsala siempre que haya una condición que haga que el resto de la app no tenga sentido (sin datos, credenciales incorrectas, etc.). Sin `st.stop()`, el código posterior intentaría crear gráficos de un DataFrame vacío y produciría errores confusos.

### 🔁 Checkpoint 2 — Probar los filtros

Guarda el archivo (la celda `%%writefile -a` ya lo hace automáticamente) y observa el navegador. Ahora verás la barra lateral con los tres `multiselect` y el slider.

**Prueba estas interacciones:**
1. Deselecciona una especie en el multiselect — la app debería seguir funcionando.
2. Deselecciona **todas** las especies — debería aparecer el mensaje de advertencia amarillo.
3. Mueve el slider de masa corporal hacia valores más restrictivos.

---

## Sección 4: Métricas Resumidas y Vista de los Datos

Los **KPIs** (*Key Performance Indicators*) son los números más importantes que el usuario quiere ver de un vistazo. Streamlit tiene el componente `st.metric` diseñado específicamente para mostrar estos valores de forma destacada, con soporte opcional para mostrar la variación respecto a un valor de referencia.

**Celda 8: Métricas KPI con `st.metric`**

In [ ]:
%%writefile -a dashboard.py

# --- Métricas KPI ---
st.subheader("📊 Resumen del Conjunto Filtrado")

# st.columns(4) crea 4 columnas de igual ancho
col1, col2, col3, col4 = st.columns(4)

with col1:
    diferencia = len(df_filtrado) - len(df)
    st.metric(
        label="Total Pingüinos",
        value=len(df_filtrado),
        delta=f"{diferencia} respecto al total"   # delta muestra flecha verde/roja
    )

with col2:
    masa_media = round(df_filtrado['body_mass_g'].mean(), 1)
    st.metric(label="Masa Corporal Media (g)", value=masa_media)

with col3:
    aleta_media = round(df_filtrado['flipper_length_mm'].mean(), 1)
    st.metric(label="Longitud de Aleta Media (mm)", value=aleta_media)

with col4:
    n_especies = df_filtrado['species'].nunique()
    st.metric(label="Especies Presentes", value=n_especies)

**Explicación — `st.columns` y `st.metric`:**

**`st.columns(n)`** divide el espacio horizontal en `n` columnas de igual ancho y devuelve una lista de objetos columna. Puedes usar `with col:` (como en el ejemplo) o llamar directamente a los métodos de la columna: `col1.metric(...)`. También puedes especificar anchos relativos: `st.columns([2, 1, 1])` crea una columna doble y dos sencillas.

**`st.metric(label, value, delta)`** muestra el valor en grande con el label encima. El parámetro `delta` es opcional — si es positivo muestra una flecha verde ↑, si es negativo una flecha roja ↓. Esto es muy útil para mostrar cambios respecto a un periodo anterior o una referencia.

**Celda 9: Tabla de Datos Interactiva**

In [ ]:
%%writefile -a dashboard.py

# --- Tabla de datos filtrados dentro de un expander ---
with st.expander("🔍 Ver datos filtrados", expanded=False):
    st.dataframe(
        df_filtrado.reset_index(drop=True),
        use_container_width=True,
        height=300
    )
    st.caption(f"Mostrando {len(df_filtrado)} registros de {len(df)} totales.")

**Explicación — `st.expander` y `st.dataframe`:**

**`st.expander(label, expanded)`** crea un contenedor colapsable. `expanded=False` hace que aparezca cerrado por defecto, manteniendo el dashboard limpio. El usuario puede expandirlo cuando quiera ver los datos crudos.

**`st.dataframe`** muestra un DataFrame de pandas como una tabla interactiva donde el usuario puede:
- Ordenar por cualquier columna haciendo clic en su cabecera
- Hacer scroll horizontal y vertical
- Buscar valores con Ctrl+F

A diferencia de `st.table()` (que genera HTML estático), `st.dataframe()` es interactivo. `use_container_width=True` hace que la tabla ocupe todo el ancho disponible. `reset_index(drop=True)` evita mostrar el índice original del DataFrame filtrado (que tendría saltos como 0, 3, 7...) y lo reemplaza por 0, 1, 2, 3...

### 🔁 Checkpoint 3 — Ver métricas y tabla

Observa las 4 tarjetas de métricas. **Cambia el filtro de especie** en la barra lateral y verifica que:
- El número total de pingüinos disminuye.
- Los valores de masa media y longitud de aleta cambian.
- El delta de la primera métrica muestra la diferencia en rojo (negativo).

Abre el expander con el botón "Ver datos filtrados" para inspeccionar los registros individuales.

---

## Sección 5: Visualizaciones Interactivas con Plotly

Streamlit soporta múltiples librerías de visualización: `st.line_chart` / `st.bar_chart` para gráficos rápidos, `st.pyplot()` para figuras de Matplotlib/Seaborn y `st.plotly_chart()` para Plotly. 

**¿Por qué Plotly en un dashboard?** A diferencia de Matplotlib, los gráficos de Plotly son **interactivos por defecto**: el usuario puede hacer zoom, paneo, pasar el ratón para ver tooltips y hacer clic en la leyenda para ocultar/mostrar series. Esto los hace ideales para dashboards exploratorios.

**Celda 10: Histograma Interactivo**

In [ ]:
%%writefile -a dashboard.py

st.divider()
st.subheader("📈 Distribuciones")

# Mapa de nombres de columna a etiquetas legibles
nombres_variables = {
    'bill_length_mm':    'Longitud del pico (mm)',
    'bill_depth_mm':     'Profundidad del pico (mm)',
    'flipper_length_mm': 'Longitud de la aleta (mm)',
    'body_mass_g':       'Masa corporal (g)'
}

col_izq, col_der = st.columns(2)

with col_izq:
    # Selector de variable para el histograma
    variable_hist = st.selectbox(
        label="Variable para el histograma",
        options=list(nombres_variables.keys()),
        format_func=lambda k: nombres_variables[k]   # muestra la etiqueta, no el nombre de columna
    )
    fig_hist = px.histogram(
        df_filtrado,
        x=variable_hist,
        color='species',
        nbins=30,
        barmode='overlay',
        opacity=0.7,
        title="Distribución por especie",
        labels={variable_hist: nombres_variables[variable_hist], 'species': 'Especie'}
    )
    fig_hist.update_layout(legend_title_text='Especie')
    st.plotly_chart(fig_hist, use_container_width=True)

**Explicación — `st.selectbox` y `px.histogram`:**

**`st.selectbox`** es un desplegable de selección única (a diferencia de `multiselect`). El parámetro `format_func` recibe una función que transforma cada opción antes de mostrársela al usuario. En este caso, el widget almacena el nombre de columna real (`bill_length_mm`) pero muestra la etiqueta legible (`'Longitud del pico (mm)'`). Esto es muy útil para separar la lógica interna de la presentación.

**`px.histogram`** con `barmode='overlay'` superpone las distribuciones de cada especie usando transparencia (`opacity=0.7`), lo que facilita la comparación visual. El parámetro `labels` permite renombrar los ejes y la leyenda directamente en la figura.

**Celda 11: Diagrama de Caja (Boxplot)**

In [ ]:
%%writefile -a dashboard.py

with col_der:
    # Selector de variable para el boxplot
    # key="boxplot_var" es necesario porque hay dos st.selectbox con el mismo label
    variable_box = st.selectbox(
        label="Variable para el diagrama de caja",
        options=list(nombres_variables.keys()),
        index=2,   # por defecto: flipper_length_mm
        format_func=lambda k: nombres_variables[k],
        key="boxplot_var"
    )
    fig_box = px.box(
        df_filtrado,
        x='species',
        y=variable_box,
        color='species',
        points='outliers',   # muestra solo los puntos outliers
        title="Distribución por especie (Boxplot)",
        labels={
            'species': 'Especie',
            variable_box: nombres_variables[variable_box]
        }
    )
    fig_box.update_layout(showlegend=False)
    st.plotly_chart(fig_box, use_container_width=True)

**Explicación — El parámetro `key`:**

Streamlit identifica internamente cada widget por su tipo y su `label`. Si tienes **dos widgets del mismo tipo con el mismo label**, Streamlit lanzará un error de duplicado. El parámetro `key` resuelve esto asignando un identificador único explícito al widget. Es una buena práctica añadir `key` siempre que uses el mismo tipo de widget múltiples veces en la misma app.

**`px.box`** con `points='outliers'` muestra los puntos individuales que caen fuera de los bigotes (1.5×IQR), lo que es mucho más informativo que ocultar todos los puntos. La alternativa `points='all'` muestra todos los puntos (útil con datasets pequeños).

**Lectura de un boxplot:** La caja central representa el rango intercuartílico (IQR = Q3 - Q1), la línea interior es la mediana (Q2), y los bigotes se extienden hasta 1.5×IQR desde los extremos de la caja.

**Celda 12: Gráfico de Dispersión Multi-variable**

In [ ]:
%%writefile -a dashboard.py

st.divider()
st.subheader("🔵 Relaciones entre Variables")

# Dos selectbox para elegir los ejes X e Y del scatter plot
col_sx, col_sy = st.columns(2)
with col_sx:
    eje_x = st.selectbox(
        "Eje X",
        list(nombres_variables.keys()),
        format_func=lambda k: nombres_variables[k],
        key="scatter_x"
    )
with col_sy:
    eje_y = st.selectbox(
        "Eje Y",
        list(nombres_variables.keys()),
        index=2,
        format_func=lambda k: nombres_variables[k],
        key="scatter_y"
    )

fig_scatter = px.scatter(
    df_filtrado,
    x=eje_x,
    y=eje_y,
    color='species',      # color por especie
    symbol='sex',         # forma del punto por sexo
    size='body_mass_g',   # tamaño del punto por masa corporal
    hover_data=['island'],   # información extra en el tooltip
    title=f"{nombres_variables[eje_x]} vs {nombres_variables[eje_y]}",
    labels={
        eje_x: nombres_variables[eje_x],
        eje_y: nombres_variables[eje_y],
        'species': 'Especie',
        'sex': 'Sexo'
    }
)
st.plotly_chart(fig_scatter, use_container_width=True)

**Explicación — Scatter plot multi-encoding:**

Este gráfico usa **cuatro canales visuales** simultáneamente para comunicar información:
- **Posición X/Y** — las dos variables numéricas seleccionadas por el usuario
- **Color** (`color='species'`) — diferencia las tres especies
- **Forma** (`symbol='sex'`) — diferencia machos y hembras
- **Tamaño** (`size='body_mass_g'`) — muestra la masa corporal de cada individuo

El parámetro `hover_data=['island']` añade la isla de cada pingüino al tooltip que aparece al pasar el ratón, sin necesidad de añadir un canal visual adicional. Esta es la filosofía de los scatter plots ricos: comunicar múltiples dimensiones en un solo gráfico compacto.

**Celda 13: Composición por Isla — Barras Apiladas y Donut**

In [ ]:
%%writefile -a dashboard.py

st.divider()
st.subheader("🏝️ Composición por Isla")

col_bar, col_pie = st.columns(2)

with col_bar:
    # Contar pingüinos por isla y especie
    conteo_isla = (
        df_filtrado
        .groupby(['island', 'species'])
        .size()
        .reset_index(name='conteo')
    )
    fig_bar = px.bar(
        conteo_isla,
        x='island',
        y='conteo',
        color='species',
        barmode='stack',   # barras apiladas
        title="Pingüinos por isla y especie",
        labels={'island': 'Isla', 'conteo': 'Número de pingüinos', 'species': 'Especie'}
    )
    st.plotly_chart(fig_bar, use_container_width=True)

with col_pie:
    # Proporción de cada especie en los datos filtrados
    conteo_especie = df_filtrado['species'].value_counts().reset_index()
    conteo_especie.columns = ['Especie', 'Conteo']
    fig_pie = px.pie(
        conteo_especie,
        names='Especie',
        values='Conteo',
        title="Proporción de especies",
        hole=0.4   # hole > 0 convierte el pie en un donut chart
    )
    st.plotly_chart(fig_pie, use_container_width=True)

**Explicación — `groupby().size()` y Donut Chart:**

El idioma `df.groupby(['col1', 'col2']).size().reset_index(name='conteo')` es una de las operaciones más frecuentes en pandas. Cuenta el número de filas para cada combinación única de valores en las columnas agrupadas, produciendo un DataFrame tidy listo para visualizar.

**Barras apiladas vs agrupadas:** `barmode='stack'` apila las barras de cada grupo encima de las otras, ideal para ver la composición total por categoría. La alternativa `barmode='group'` coloca las barras una al lado de la otra, mejor para comparar valores absolutos.

**Donut chart:** El parámetro `hole=0.4` en `px.pie()` crea un "donut" dejando un espacio vacío en el centro. Los donut charts son preferidos en dashboards profesionales sobre los pie charts tradicionales porque el ojo humano compara áreas de forma más precisa cuando hay un punto de referencia central.

### 🔁 Checkpoint 4 — Interactividad de Plotly

Ahora el dashboard tiene 5 gráficos. Prueba la interactividad nativa de Plotly:

1. **Leyenda:** Haz clic en "Adelie" en la leyenda del histograma para ocultar/mostrar esa especie.
2. **Zoom:** Dibuja un rectángulo con el ratón sobre el scatter plot para hacer zoom en esa zona.
3. **Tooltip:** Pasa el ratón sobre los puntos del scatter plot y observa la información del tooltip (incluyendo la isla).
4. **Selectbox dinámico:** Cambia los ejes X e Y del scatter plot y observa cómo el gráfico se actualiza.

---

## Sección 6: Análisis Estadístico

Un dashboard profesional no solo muestra gráficos: también proporciona herramientas de análisis cuantitativo. El mapa de calor de correlaciones y la tabla de estadísticas descriptivas son dos elementos presentes en prácticamente cualquier dashboard de EDA.

**Celda 14: Mapa de Calor de Correlaciones**

In [ ]:
%%writefile -a dashboard.py

st.divider()
st.subheader("🌡️ Mapa de Correlaciones")

# Columnas numéricas del dataset
columnas_numericas = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
etiquetas_cortas = {
    'bill_length_mm':    'Long. pico',
    'bill_depth_mm':     'Prof. pico',
    'flipper_length_mm': 'Long. aleta',
    'body_mass_g':       'Masa corporal'
}

# Calcular la matriz de correlación de Pearson
matriz_corr = df_filtrado[columnas_numericas].corr()
# Renombrar filas y columnas con etiquetas legibles
matriz_corr.index   = [etiquetas_cortas[c] for c in columnas_numericas]
matriz_corr.columns = [etiquetas_cortas[c] for c in columnas_numericas]

fig_heatmap = px.imshow(
    matriz_corr,
    text_auto='.2f',         # muestra el valor numérico dentro de cada celda
    color_continuous_scale='RdBu_r',   # escala divergente: rojo=negativo, azul=positivo
    zmin=-1, zmax=1,         # anclar la escala de color entre -1 y 1
    title="Correlación de Pearson entre variables morfológicas",
    aspect='auto'
)
st.plotly_chart(fig_heatmap, use_container_width=True)

**Explicación — Correlación de Pearson y `px.imshow`:**

**`DataFrame.corr()`** calcula la **correlación de Pearson** entre todas las parejas de columnas numéricas. El resultado es una matriz cuadrada donde el valor `(i, j)` indica qué tan linealmente relacionadas están las variables `i` y `j`:
- **+1**: correlación positiva perfecta (cuando una sube, la otra sube proporcionalmoente)
- **0**: sin correlación lineal
- **-1**: correlación negativa perfecta (cuando una sube, la otra baja)

**`px.imshow`** visualiza matrices como imágenes con colores. Los parámetros clave son:
- `text_auto='.2f'` — muestra los valores numéricos con 2 decimales dentro de cada celda
- `color_continuous_scale='RdBu_r'` — escala divergente: rojo intenso para correlaciones negativas, azul intenso para positivas, blanco para cero
- `zmin=-1, zmax=1` — ancla la escala entre -1 y 1, independientemente de los valores reales, para mantener la referencia visual consistente

**Interpretación:** En el dataset completo, la longitud de aleta y la masa corporal tienen correlación positiva fuerte (~0.87), lo que tiene sentido biológicamente: los pingüinos más grandes tienen aletas más largas.

**Celda 15: Tabla de Estadísticas Descriptivas**

In [ ]:
%%writefile -a dashboard.py

st.divider()
st.subheader("📋 Estadísticas Descriptivas")

# describe() devuelve count, mean, std, min, 25%, 50%, 75%, max
stats = df_filtrado[columnas_numericas].describe().round(2)

# Traducir el índice (nombres de filas) al español
stats.index = ['Conteo', 'Media', 'Desv. Est.', 'Mínimo', 'Q1 (25%)', 'Mediana', 'Q3 (75%)', 'Máximo']

# Traducir las columnas a etiquetas cortas
stats.columns = [etiquetas_cortas[c] for c in columnas_numericas]

st.dataframe(stats, use_container_width=True)

**Explicación — `DataFrame.describe()`:**

`describe()` calcula automáticamente las estadísticas descriptivas más importantes para cada columna numérica:

| Estadístico | Descripción |
|-------------|-------------|
| count | Número de valores no nulos |
| mean | Media aritmética |
| std | Desviación estándar |
| min | Valor mínimo |
| 25% (Q1) | Primer cuartil: el 25% de los datos está por debajo |
| 50% (mediana) | Segundo cuartil: el 50% de los datos está por debajo |
| 75% (Q3) | Tercer cuartil: el 75% de los datos está por debajo |
| max | Valor máximo |

Traducir el índice y las columnas al español hace la tabla más profesional y accesible para usuarios no técnicos.

**Celda 16: Pie de Página**

In [ ]:
%%writefile -a dashboard.py

# --- Pie de página con atribución del dataset ---
st.divider()
st.caption(
    "📚 Datos: Gorman KB, Williams TD, Fraser WR (2014). "
    "Palmer Penguins Dataset — Palmer Station, Antarctica LTER. "
    "| Laboratorio Streamlit — Máster en IA / Deep Learning."
)

**Explicación:**

La atribución de datos es una buena práctica en cualquier visualización profesional. `st.caption()` genera texto pequeño y gris, ideal para notas legales, referencias y pie de página que no deben competir visualmente con el contenido principal del dashboard.

El dashboard está completo. Continúa con la siguiente sección para verificar el archivo generado y ejecutar la versión final.

---
## Sección 7: Análisis con Inteligencia Artificial

Una de las aplicaciones más prácticas de los LLMs en dashboards es la **interpretación automática de datos**: en lugar de que el usuario tenga que leer tablas y gráficos, puede pulsar un botón y recibir un análisis en lenguaje natural generado por GPT.

En esta sección añadiremos un botón que manda las estadísticas del subconjunto **actualmente filtrado** a GPT-4o-mini y muestra el análisis directamente en el dashboard. El resultado se guarda en `st.session_state` para que no desaparezca cuando el usuario interactúa con otros widgets.

**Celda 17: Análisis del Subconjunto Filtrado con OpenAI**

In [ ]:
%%writefile -a dashboard.py

from openai import OpenAI

st.divider()
st.subheader("🤖 Análisis con Inteligencia Artificial")
st.markdown("Pulsa el botón para que GPT-4o-mini interprete el subconjunto de datos filtrado.")

# El alumno pega su API key directamente en la barra lateral
api_key = st.sidebar.text_input(
    "🔑 OpenAI API Key",
    type="password",
    placeholder="sk-..."
)

if st.button("✨ Generar análisis con IA"):
    if not api_key:
        st.warning("⚠️ Introduce tu OpenAI API Key en el panel lateral izquierdo.")
    else:
        # Preparar estadísticas del subconjunto filtrado
        columnas_num = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
        nombres_es = {
            'bill_length_mm': 'longitud pico (mm)',
            'bill_depth_mm': 'profundidad pico (mm)',
            'flipper_length_mm': 'longitud aleta (mm)',
            'body_mass_g': 'masa corporal (g)'
        }
        stats = df_filtrado[columnas_num].rename(columns=nombres_es).describe().round(1)
        conteo_especies = df_filtrado['species'].value_counts().to_string()
        conteo_islas    = df_filtrado['island'].value_counts().to_string()

        prompt = f"""Eres un experto en biología marina. Analiza en español los siguientes datos
de pingüinos del Archipiélago Palmer (subconjunto filtrado: {len(df_filtrado)} individuos).

Distribución por especie:
{conteo_especies}

Distribución por isla:
{conteo_islas}

Estadísticas morfológicas:
{stats.to_string()}

Proporciona un análisis conciso (3-4 frases) en español destacando los patrones más relevantes."""

        with st.spinner("Analizando con GPT-4o-mini..."):
            client = OpenAI(api_key=api_key)
            respuesta = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}]
            )
            analisis = respuesta.choices[0].message.content

        st.session_state['analisis_ia'] = analisis

# Mostrar el análisis guardado (persiste al interactuar con otros widgets)
if 'analisis_ia' in st.session_state:
    st.info(st.session_state['analisis_ia'])
    st.caption("Generado con GPT-4o-mini · Cambia los filtros y vuelve a pulsar para actualizar.")


**Explicación — Patrón LLM + Dashboard:**

**`st.sidebar.text_input(..., type="password")`** es la forma más sencilla de pedir una API key: el alumno la pega directamente en la barra lateral sin configurar variables de entorno. El parámetro `type="password"` oculta los caracteres.

El **prompt** se construye dinámicamente con las estadísticas del `df_filtrado` en ese momento. Cada vez que el usuario cambia un filtro y vuelve a pulsar el botón, GPT recibe datos distintos y produce un análisis diferente.

**`st.session_state['analisis_ia']`** guarda el resultado. Sin esto, el análisis desaparecería en cada rerun (cada vez que el usuario interactúa con un widget). El bloque `if 'analisis_ia' in st.session_state` al final lo re-renderiza en todos los reruns posteriores.

**`st.info(texto)`** muestra el análisis en un recuadro azul, diferenciándolo visualmente del resto del dashboard.

> 💡 **Coste estimado:** GPT-4o-mini consume ~400 tokens por análisis. El coste es inferior a $0.001 por llamada.

### 🔁 Checkpoint 5 — Probar el análisis con IA

1. Ejecuta la app y pulsa el botón **"✨ Generar análisis con IA"**.
2. Cambia el filtro de especie en el sidebar (por ejemplo, selecciona solo *Gentoo*).
3. Pulsa el botón de nuevo — el análisis debe ser diferente porque los datos han cambiado.
4. Interactúa con otros widgets (slider, gráficos) — el análisis anterior debe seguir visible gracias a `session_state`.

---

---
## Sección 8: Ejecución Final, Bonus y Reflexión

**Celda 17: Verificación del Archivo Completo**

In [ ]:
import os

ruta = 'dashboard.py'
if os.path.exists(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        lineas = f.readlines()
    print(f"✅ Archivo '{ruta}' creado correctamente.")
    print(f"   Total de líneas: {len(lineas)}")
    print("\n--- Primeras 15 líneas ---")
    for i, linea in enumerate(lineas[:15], 1):
        print(f"{i:3}: {linea}", end='')
else:
    print(f"❌ ERROR: El archivo '{ruta}' no existe.")
    print("   Vuelve a ejecutar todas las celdas desde la Celda 3.")

**Celda 18: Instrucciones de Ejecución Final**

El archivo `dashboard.py` está completo. Para lanzar el dashboard en su versión definitiva:

1. Abre una terminal en la misma carpeta que `dashboard.py`.
2. Ejecuta el siguiente comando:

```bash
streamlit run dashboard.py
```

3. Streamlit abrirá automáticamente tu navegador en `http://localhost:8501`
4. Explora el dashboard completo con todos los filtros y gráficos.

**Tip avanzado:** Añade `-- --server.runOnSave true` para que la app recargue automáticamente cada vez que guardes el archivo:
```bash
streamlit run dashboard.py --server.runOnSave true
```

Para detener la aplicación, presiona `Ctrl + C` en la terminal.

**Celda 19 (Bonus): Session State — Persistencia de Estado entre Reruns**

In [ ]:
%%writefile bonus_state.py
# BONUS: Ejemplo de st.session_state para persistir valores entre reruns
# Ejecuta con: streamlit run bonus_state.py

import streamlit as st

st.title("Ejemplo: Session State")
st.markdown("""
Streamlit re-ejecuta el script completo de arriba a abajo cada vez que el usuario
interactúa con un widget. Esto significa que las variables locales se reinician
en cada interacción. `st.session_state` es un diccionario especial que **persiste
entre reruns** dentro de la misma sesión del usuario.
""")

# Inicializar el contador en el estado de sesión si no existe
if 'contador' not in st.session_state:
    st.session_state.contador = 0

# Tres botones para modificar el estado
col1, col2, col3 = st.columns(3)
with col1:
    if st.button("➕ Incrementar"):
        st.session_state.contador += 1
with col2:
    if st.button("➖ Decrementar"):
        st.session_state.contador -= 1
with col3:
    if st.button("🔄 Reiniciar"):
        st.session_state.contador = 0

st.metric("Valor actual del contador", st.session_state.contador)
st.info(
    "Cada clic en un botón provoca un rerun completo del script. "
    "Sin session_state, el contador volvería a 0 en cada interacción."
)

**Explicación — El modelo de ejecución de Streamlit:**

Esta es la pieza conceptual más importante para entender Streamlit:

> **Cada vez que el usuario interactúa con cualquier widget, Streamlit re-ejecuta el script Python completo de arriba a abajo.**

Esto tiene una implicación fundamental: **todas las variables locales se pierden entre reruns**. Si tuvieras un contador como variable local Python (`contador = 0`), se reiniciaría a 0 en cada interacción.

**`st.session_state`** es un diccionario persistente que sobrevive a los reruns dentro de la misma sesión del navegador. Para usarlo:
1. Inicializa las claves con `if 'clave' not in st.session_state: st.session_state.clave = valor_inicial`
2. Lee el valor con `st.session_state.clave`
3. Modifícalo desde cualquier callback de botón o widget

Aunque en el dashboard de pingüinos no necesitamos session_state (los filtros del sidebar son suficientes), es imprescindible para apps más complejas: formularios de múltiples pasos, carritos de compra, conversaciones con modelos de lenguaje, etc.

Prueba este ejemplo ejecutando `streamlit run bonus_state.py` en la terminal.

## Ejercicio Propuesto: Amplía el Dashboard

Ahora que dominas los componentes básicos de Streamlit, intenta añadir al menos **dos** de las siguientes mejoras al archivo `dashboard.py`:

### Nivel básico
1. **Descarga de datos** — Añade un botón `st.download_button` que permita descargar el DataFrame filtrado como CSV.
   ```python
   csv = df_filtrado.to_csv(index=False).encode('utf-8')
   st.download_button("📥 Descargar CSV", data=csv, file_name="pinguinos_filtrados.csv", mime="text/csv")
   ```

2. **Violin plot** — Añade un `px.violin` como alternativa al boxplot para ver la distribución completa de los datos.
   ```python
   fig_violin = px.violin(df_filtrado, x='species', y='flipper_length_mm', color='species', box=True)
   ```

### Nivel intermedio
3. **Pestañas** — Organiza el contenido en pestañas usando `st.tabs(['📊 Dashboard', '📋 Datos', 'ℹ️ Acerca del Dataset'])`.

4. **Scatter plot 3D** — Usa `px.scatter_3d` para visualizar tres variables morfológicas simultáneamente.
   ```python
   fig_3d = px.scatter_3d(df_filtrado, x='bill_length_mm', y='bill_depth_mm', z='flipper_length_mm', color='species')
   ```

### Nivel avanzado
5. **Modo oscuro dinámico** — Añade un `st.sidebar.toggle('Modo oscuro')` y cambia el template de Plotly según el modo:
   ```python
   template = 'plotly_dark' if modo_oscuro else 'plotly_white'
   ```

6. **Histograma animado** — Usa `px.histogram` con el parámetro `animation_frame='species'` para crear un histograma animado por especie.

---
**Consulta la documentación oficial:** https://docs.streamlit.io/library/api-reference

## Conclusiones

En este laboratorio has construido un dashboard de datos profesional e interactivo con Streamlit. Estos son los componentes que has aprendido a usar:

### Configuración y datos
| Componente | Uso |
|-----------|-----|
| `st.set_page_config` | Título, icono y layout de la app |
| `@st.cache_data` | Caché de datos para evitar recargas innecesarias |
| `st.stop()` | Detener la ejecución cuando no hay datos |

### Texto y estructura
| Componente | Uso |
|-----------|-----|
| `st.title / subheader / markdown` | Jerarquía de texto |
| `st.columns(n)` | Layout en columnas |
| `st.expander` | Contenedor colapsable |
| `st.divider` | Separador visual |

### Widgets interactivos
| Componente | Uso |
|-----------|-----|
| `st.sidebar.multiselect` | Selección múltiple de categorías |
| `st.sidebar.slider` | Rango numérico |
| `st.selectbox` | Selección única con etiquetas personalizadas |

### Visualización y datos
| Componente | Uso |
|-----------|-----|
| `st.metric` | KPI con delta opcional |
| `st.dataframe` | Tabla interactiva sortable |
| `st.plotly_chart` | Gráficos interactivos de Plotly |

### Conceptos clave
- **Modelo de re-ejecución:** Streamlit re-ejecuta el script completo en cada interacción.
- **`st.session_state`:** Para persistir valores entre reruns.
- **`@st.cache_data`:** Para evitar recargar datos costosos en cada rerun.
- **Parámetro `key`:** Obligatorio cuando hay múltiples widgets del mismo tipo con el mismo label.

Streamlit es ampliamente usado en la industria para **prototipar aplicaciones de ML**, **presentar modelos a stakeholders** y **explorar datos de forma interactiva**. Con lo aprendido en este laboratorio tienes las bases para construir cualquier dashboard de datos que necesites.

---
### 🐧 ¡Felicidades! Has completado el Laboratorio de Streamlit.